# 🇮🇳 VoiceGuard India - The HuggingFace Architecture (Disk Download)

### ⚠️ CRITICAL: You MUST unlock these datasets manually first!
Before running this notebook, you must click these links, log into HuggingFace, and click **Agree to Terms of Use** or **Request Access** on their pages.

1. **IndicSynth (Deepfakes):** [https://huggingface.co/datasets/vdivyasharma/IndicSynth](https://huggingface.co/datasets/vdivyasharma/IndicSynth)
2. **Abhirl TTS (Deepfakes):** [https://huggingface.co/datasets/abhirl/hindi-tts-dataset](https://huggingface.co/datasets/abhirl/hindi-tts-dataset)
3. **UD-NLP Telephone (Authentic):** [https://huggingface.co/datasets/ud-nlp/hindi-speech-recognition-dataset](https://huggingface.co/datasets/ud-nlp/hindi-speech-recognition-dataset)

In [ ]:
!pip install datasets librosa scikit-learn numpy tqdm xgboost soundfile

In [ ]:
import os
import librosa
import numpy as np
import pickle
from datasets import load_dataset
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# ⚠️ ENTER YOUR HUGGINGFACE TOKEN HERE ⚠️
# ==========================================
HF_TOKEN = "hf_YOUR_TOKEN_HERE"  

def safe_load(path, *args, **kwargs):
    try:
        # FIX: Removed streaming=True. We are downloading straight to the 73GB Kaggle Disk!
        return load_dataset(path, *args, token=HF_TOKEN, **kwargs)
    except Exception as e:
        print(f"⚠️ Error loading {path}: {e}")
        return None

In [ ]:
print("🌍 Downloading HuggingFace Datasets to Kaggle Disk (This will be much faster than streaming)...")

# -- AUTHENTIC HUMAN HINDI STREAMS (Label 1) --
auth_1 = safe_load("google/fleurs", "hi_in", split="train")
auth_2 = safe_load("SPRINGLab/IndicTTS-Hindi", split="train")
auth_3 = safe_load("ud-nlp/hindi-speech-recognition-dataset", split="train")

# -- DEEPFAKE HINDI STREAMS (Label 0) --
fake_1 = safe_load("vdivyasharma/IndicSynth", "Hindi", split="train")
fake_2 = safe_load("abhirl/hindi-tts-dataset", split="train")

print("\n✅ Datasets Successfully Downloaded!")

In [ ]:
def extract_features_from_array(audio_array, sr):
    try:
        mfccs = np.mean(librosa.feature.mfcc(y=audio_array, sr=sr, n_mfcc=20).T, axis=0)
        centroid = np.mean(librosa.feature.spectral_centroid(y=audio_array, sr=sr))
        zcr = np.mean(librosa.feature.zero_crossing_rate(audio_array))
        return np.hstack((mfccs, centroid, zcr))
    except:
        return None

X = []
y = []
SAMPLES_PER_DATASET = 2000 # Increased stability for memory limits

def consume_dataset(dataset, label, name, max_samples):
    if dataset is None: return
    print(f"\n⏬ Extracting Features from: {name}")
    count = 0
    try:
        for item in tqdm(dataset, total=max_samples):
            if count >= max_samples: break
            if 'audio' in item and 'array' in item['audio']:
                features = extract_features_from_array(item['audio']['array'], item['audio']['sampling_rate'])
                if features is not None:
                    X.append(features)
                    y.append(label)
                    count += 1
    except Exception as e:
        print(f"⚠️ Error extracting {name}: {e}")

# Consume Authentic Streams
consume_dataset(auth_1, 1, "Google Fleurs (Hindi)", SAMPLES_PER_DATASET)
consume_dataset(auth_2, 1, "IndicTTS-Hindi", SAMPLES_PER_DATASET)
consume_dataset(auth_3, 1, "UD-NLP Hindi Telephone", SAMPLES_PER_DATASET)

# Consume Fake Streams
consume_dataset(fake_1, 0, "IndicSynth Deepfakes", SAMPLES_PER_DATASET)
consume_dataset(fake_2, 0, "Hindi TTS Wikipedia Clones", SAMPLES_PER_DATASET)

X_final = np.array(X)
y_final = np.array(y)

if len(np.unique(y_final)) < 2:
    print("\n\n❌ FATAL ERROR: You are missing either Authentic or Deepfake data! Make sure you unlocked the HuggingFace datasets online.")
else:
    print(f"\n✅ Successfully extracted features for {len(X_final)} total audio files!")

In [ ]:
if len(np.unique(y_final)) >= 2:
    print("\n🚀 Training XGBoost Classifier...")
    clf = xgb.XGBClassifier(n_estimators=300, max_depth=10, learning_rate=0.05, n_jobs=-1, random_state=42)
    clf.fit(X_final, y_final)
    
    print("\n✅ Training Complete! Accuracy Metrics:")
    predictions = clf.predict(X_final)
    print(classification_report(y_final, predictions, target_names=["Deepfake", "Authentic"]))
    
    with open("voiceguard_ultimate_model.pkl", "wb") as f:
        pickle.dump(clf, f)
        
    print("\n💾 SUCCESS! Ultimate Model saved as 'voiceguard_ultimate_model.pkl'")